# FarmGemma Vision Training - Google Colab

Fine-tune vision encoder for crop disease detection

**Runtime**: T4 GPU - Free!

In [ ]:
# Install dependencies
!pip install -q transformers datasets peft accelerate pillow
!pip install -q torch torchvision
!pip install -q huggingface_hub

print("✅ Dependencies installed!")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR = "/content/drive/MyDrive/farmgemma"
DATA_DIR = "/content/drive/MyDrive/farmgemma/data"
import os
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
# Download PlantVillage dataset (crop disease images)
from huggingface_hub import snapshot_download

print("📥 Downloading PlantVillage dataset...")
snapshot_download(
    repo_id="AlifMiah/PlantVillage-Dataset",
    local_dir=f"{DATA_DIR}/PlantVillage",
    ignore_patterns=["*.zip"]
)
print("✅ Dataset downloaded!")

In [ ]:
# Load and prepare vision model (SigLIP)
import torch
from transformers import AutoModel, AutoProcessor
from PIL import Image
import os

print("📥 Loading SigLIP vision model...")
vision_model = AutoModel.from_pretrained(
    "google/siglip-base-patch16-224",
    torch_dtype=torch.bfloat16
)
processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
print("✅ Vision model loaded!")

In [ ]:
# Create image classification dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class CropDiseaseDataset(Dataset):
    def __init__(self, data_dir, processor):
        self.processor = processor
        self.samples = []
        self.labels = []
        
        # Build label index
        classes = sorted(os.listdir(data_dir))
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        
        for class_name in classes:
            class_dir = os.path.join(data_dir, class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir)[:500]:  # Limit per class
                    if img_name.endswith(('.jpg', '.png')):
                        self.samples.append({
                            'image_path': os.path.join(class_dir, img_name),
                            'label': self.class_to_idx[class_name]
                        })
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample['image_path']).convert('RGB')
        
        inputs = self.processor(images=image, return_tensors="pt")
        inputs['labels'] = torch.tensor(sample['label'])
        
        return inputs

# Create dataset
dataset_path = f"{DATA_DIR}/PlantVillage/raw/PlantVillage-Dataset"
dataset = CropDiseaseDataset(dataset_path, processor)
print(f"✅ Dataset created: {len(dataset)} images, {len(dataset.class_to_idx)} classes")

In [ ]:
# Setup LoRA for vision model
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)

vision_model = get_peft_model(vision_model, lora_config)
vision_model.print_trainable_parameters()

In [ ]:
# Training loop
from torch.optim import AdamW
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vision_model = vision_model.to(device)
vision_model.train()

optimizer = AdamW(vision_model.parameters(), lr=1e-4)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

epochs = 3
for epoch in range(epochs):
    total_loss = 0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for batch in pbar:
        pixel_values = batch['pixel_values'].squeeze(1).to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        outputs = vision_model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}: Average Loss = {avg_loss:.4f}")

In [ ]:
# Save fine-tuned vision model
vision_model.save_pretrained(f"{MODEL_DIR}/farmgemma-vision")
processor.save_pretrained(f"{MODEL_DIR}/farmgemma-vision")
print(f"✅ Vision model saved to {MODEL_DIR}/farmgemma-vision")

In [ ]:
# Test the model
from io import BytesIO

# Download a test image
!wget -q -O /tmp/test_leaf.jpg https://raw.githubusercontent.com/spMohanty/PlantVillage-Dataset/master/raw/segmented/Apple___Apple_scab/image.JPG-105.jpg

test_image = Image.open("/tmp/test_leaf.jpg").convert('RGB')
inputs = processor(images=test_image, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = vision_model(**inputs)
    logits = outputs.logits
    predicted_class = logits.argmax(-1).item()

# Get class name
idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}
print(f"� Diagnosis: {idx_to_class[predicted_class]}")

## Next Steps

1. **Combine** vision + LLM for multimodal FarmGemma
2. **Merge** with text-only fine-tuned model
3. **Quantize** for edge deployment